In [ ]:
# Eval Regression – q3 vs IQR/RMS, residuals by energy, IQR vs training samples (bin 2)
# Plots saved as PDFs in out/regression_eval

from sklearn.metrics import confusion_matrix
import seaborn as sns

from eval_classification_plots import (
    load_results,
    load_truth_and_baselines,
    data_with_signal_pion_bins,
    compute_all_metrics,
    compute_all_metrics_q3,
    compute_signal_baseline,
    compute_reco_baseline_recall_per_bin,
    plot_cc1pi_vs_pion_kinematics,
    plot_multi_pion_vs_q3,
    plot_binned_by_inttype,
    plot_prc_curves,
    save_figures_to_pdf,
    CLASSIFICATION_PERFORMANCE_LEGEND_TITLE,
)

# Performance plots use CLASSIFICATION_PERFORMANCE_LEGEND_TITLE as legend title (plot_* defaults).
_ = CLASSIFICATION_PERFORMANCE_LEGEND_TITLE

import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

CKPT_DIR = Path("/global/cfs/cdirs/m3246/gregork/checkpoints")
OUTPUT_DIR = ROOT / "out" / "classification_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WANDB_TAG = "Run_2703"  # wandb tag to select runs (set as needed)
BINS_E_PION = [0, 0.4, 0.8, 1.2, 1.6, 2.0, 2.2]


## FLOPs vs validation loss (classification)

Plot log10(cumulative FLOPs) vs validation loss for each classification model (full-dataset runs), using eval_loss logged to wandb every 1000 steps.

The following section adds validation loss vs log10(training step), matching Eval_Regression.ipynb.


In [ ]:
# FLOPs per step (BS 2048) – same as regression
flops_per_step = {
    "Transformer-xsmall": 358.5 * 1e9,
    "OmniLearned-small": 1769 * 1e9,
    "MLP": 2.6 * 1e9,
    "OmniLearned-medium": 8035 * 1e9,
    "OmniLearned-small-rw": 1769 * 1e9,
    "Transformer2": 6236 * 1e9,
    "Transformer-small": 1263 * 1e9
}


clrs_dict_full = {
    "Transformer": "#1f77b4",          # a nice blue
    "Transformer-xsmall": "#1f77b4",    # a nice blue
    "OmniLearned-small": "#ff7f0e",    # a vibrant orange
    "Transformer-small": "#17becf",
    "MLP": "#2ca02c",                  # a pleasant green
    "OmniLearned-medium": "#9467bd",   # a pretty purple
    "OmniLearned-small-rw": "#e377c2", # a distinct pink/magenta, distant from others
    "Transformer2": "#d62728"
}

def get_validation_loss_history(run_name, project="minerva-models", with_steps=False):
    """Return array of all logged eval_loss values for a run (from wandb history).
    If with_steps=True, return (steps, losses) for x-axis (e.g. cumulative flops)."""
    import os
    import numpy as np
    import wandb
    api = wandb.Api(timeout=60)
    entity = os.environ.get("WANDB_ENTITY") or getattr(api, "default_entity", None)
    if not entity:
        raise RuntimeError("WANDB_ENTITY not set and wandb default_entity unknown. Run wandb login or set WANDB_ENTITY.")
    path = f"{entity}/{project}"
    runs = api.runs(path, filters={"displayName": run_name})
    run = next(iter(runs), None)
    if run is None:
        raise ValueError(f"Wandb run not found: {run_name!r} in {path}")
    hist = run.history()
    if "eval_loss" not in hist.columns:
        return (np.array([]), np.array([])) if with_steps else np.array([])
    mask = hist["eval_loss"].notna()
    steps = np.asarray(hist["_step"], dtype=float)[mask]
    losses = np.asarray(hist["eval_loss"].dropna(), dtype=float)
    # Keep only every 1000th step (1000, 2000, 3000, ...) as data points
    step_int = np.round(steps).astype(int)
    keep = (step_int >= 1000) & (step_int % 1000 == 0)
    steps, losses = steps[keep], losses[keep]
    # Sort by step and take first occurrence per step (in case of duplicates)
    order = np.argsort(steps)
    steps, losses = steps[order], losses[order]
    _, idx = np.unique(np.round(steps).astype(int), return_index=True)
    steps = steps[np.sort(idx)]
    losses = losses[np.sort(idx)]
    if with_steps:
        return steps, losses
    return losses


In [ ]:
from src.utils.utils import get_classification_runs_by_model_and_cap

runs_by_model_cap = get_classification_runs_by_model_and_cap(WANDB_TAG)

training_names = {
    key: value[-1]
    for key, value in sorted(runs_by_model_cap.items(), key=lambda kv: kv[0])
}


In [ ]:
# Log flops vs validation loss: one curve per model; data points every 1000 steps; ±1 sigma band across runs (when available)
import numpy as np
import matplotlib.pyplot as plt

# Runs per model for full dataset (cap=-1); use all seeds when available
runs_per_model = {}
for model in sorted(flops_per_step):
    if model not in runs_by_model_cap or -1 not in runs_by_model_cap[model]:
        run_list = training_names.get(model)
        if run_list is not None:
            runs_per_model[model] = run_list if isinstance(run_list, list) else [run_list]
        continue
    runs_per_model[model] = runs_by_model_cap[model][-1]

loss_plot_legend_title = (
    "Minerva Open Data Playlist 1A\n"
    "Task: Classification"
)

fig, ax = plt.subplots(figsize=(8, 5))

for model, run_names in sorted(runs_per_model.items(), key=lambda kv: kv[0]):
    color = clrs_dict_full[model]
    flops = flops_per_step[model]
    all_steps, all_losses = [], []
    for rn in run_names:
        st, lo = get_validation_loss_history(rn, with_steps=True)  # already every 1000 steps
        if len(st) > 0 and len(lo) > 0:
            all_steps.append(st)
            all_losses.append(lo)

    if not all_steps:
        continue

    # Full range: union of all steps across runs (no truncation to minimum)
    steps_grid = np.unique(np.concatenate(all_steps)).astype(float)
    if len(steps_grid) == 0:
        continue
    # Interpolate each run onto the common grid (extrapolates with edge values beyond run range)
    losses_aligned = np.array([
        np.interp(steps_grid, st, lo) for st, lo in zip(all_steps, all_losses)
    ])

    mean_loss = np.mean(losses_aligned, axis=0)
    # Std across runs only (when multiple runs); no band for single run
    sigma_loss = np.std(losses_aligned, axis=0) if losses_aligned.shape[0] > 1 else np.zeros_like(mean_loss)

    cum_flops = steps_grid * flops
    x = np.log10(cum_flops + 1)  # +1 to avoid log(0) at step 0
    ax.plot(x, mean_loss, color=color, label=model)
    if losses_aligned.shape[0] > 1:
        ax.fill_between(x, mean_loss - sigma_loss, mean_loss + sigma_loss, alpha=0.25, color=color)

ax.set_xlabel(f"$log_{{10}}$(Training FLOPs)")
ax.set_ylabel("Validation loss")
ax.set_ylim([1.075, 1.25])

leg_flops = ax.legend(title=loss_plot_legend_title, fontsize=9, loc="upper right")
leg_flops.set_title(loss_plot_legend_title)
ax.grid(True)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "log_flops_vs_val_loss.pdf", bbox_inches="tight")
print("Saved:", OUTPUT_DIR / "log_flops_vs_val_loss.pdf")
plt.show()


## Validation loss vs training step (classification)

Same idea as the regression notebook: `eval_loss` logged every 1000 steps vs log10(training step), one curve per model (mean ±1σ across seeds when multiple runs exist). OmniLearned-medium curves are truncated at 25k steps to match the regression plot.


In [ ]:
# Plot: Validation loss vs log10(training step), with custom cut for OLM models (same as Eval_Regression)
loss_plot_legend_title = (
    "Minerva Open Data Playlist 1A\n"
    "Task: Classification"
)

fig2, ax2 = plt.subplots(figsize=(8, 5))

for model, run_names in sorted(runs_per_model.items(), key=lambda kv: kv[0]):
    all_steps, all_losses = [], []
    color = clrs_dict_full[model]
    for rn in run_names:
        st, lo = get_validation_loss_history(rn, with_steps=True)
        if len(st) > 0 and len(lo) > 0:
            all_steps.append(st)
            all_losses.append(lo)

    if not all_steps:
        continue

    steps_grid = np.unique(np.concatenate(all_steps)).astype(float)
    if len(steps_grid) == 0:
        continue

    losses_aligned = np.array([
        np.interp(steps_grid, st, lo) for st, lo in zip(all_steps, all_losses)
    ])
    mean_loss = np.mean(losses_aligned, axis=0)
    sigma_loss = np.std(losses_aligned, axis=0) if losses_aligned.shape[0] > 1 else np.zeros_like(mean_loss)

    if "OmniLearned-medium" in model:
        step_mask = steps_grid <= 25_000
    else:
        step_mask = np.full_like(steps_grid, True, dtype=bool)
    steps_plot = steps_grid[step_mask]
    mean_loss_plot = mean_loss[step_mask]
    sigma_loss_plot = sigma_loss[step_mask]

    log_steps_plot = np.log10(steps_plot + 1)
    ax2.plot(log_steps_plot, mean_loss_plot, color=color, label=model)
    if losses_aligned.shape[0] > 1:
        ax2.fill_between(log_steps_plot, mean_loss_plot - sigma_loss_plot, mean_loss_plot + sigma_loss_plot, alpha=0.25, color=color)

ax2.set_xlabel(f"$log_{{10}}$(Training steps)")
ax2.set_ylabel("Validation loss")
ax2.set_ylim([1.075, 1.25])
leg_steps = ax2.legend(title=loss_plot_legend_title, fontsize=9, loc="upper right")
leg_steps.set_title(loss_plot_legend_title)
ax2.grid(True)
fig2.tight_layout()
fig2.savefig(OUTPUT_DIR / "log_steps_vs_val_loss.pdf", bbox_inches="tight")
print("Saved:", OUTPUT_DIR / "log_steps_vs_val_loss.pdf")
plt.show()


In [ ]:
PLAYLISTS = ["1A", "1B"]
results = load_results(CKPT_DIR, training_names, playlists=PLAYLISTS)
data_by_playlist = {
    pl: load_truth_and_baselines(CKPT_DIR, training_names, playlists=[pl])
    for pl in PLAYLISTS
}
# For backward compatibility, data is 1A (used in cells that are not yet looped)
data = data_by_playlist["1A"]

## Confusion matrix

In [ ]:

class_names = ["CC1π±", "CCNπ", "CC1π0", "Other-CC", "Other-NC"]
model_names = sorted(training_names.keys())
n_models = len(model_names)

for playlist in PLAYLISTS:
    fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6), tight_layout=True)
    if n_models == 1:
        axes = [axes]
    for ax, model_name in zip(axes, model_names):
        run0 = results[model_name][0][playlist]
        y_true = run0["pid"]
        y_pred = run0["prediction"].argmax(axis=1)
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=class_names, yticklabels=class_names)
        ax.set_xlabel(r"Predicted class")
        ax.set_ylabel(r"True class")
        ax.set_title(model_name)
    fig.suptitle(f"Confusion matrices (first run per model) — {playlist}", fontsize=14)
    fig.savefig(OUTPUT_DIR / f"confusion_matrices_{playlist}.pdf")
    fig.show()

## CC1π± tagging

In [ ]:
cc1pi_classes = [0]

for playlist in PLAYLISTS:
    data = data_by_playlist[playlist]
    figs_cc1pi = []

    # --- Reconstruction baseline ---
    test_idx = data["test_idx"][playlist]
    baselines_pl = data["baselines"][playlist]

    n_muons = baselines_pl["n_muons"][test_idx]
    n_charged_prongs = baselines_pl["n_charged_prongs"][test_idx]
    improved_nmichel = baselines_pl["improved_nmichel"][test_idx]

    first_model = next(iter(results))
    run0 = results[first_model][0][playlist]
    pid = run0["pid"]
    data_cc1pi = data_with_signal_pion_bins(
        data, pid, cc1pi_classes,
        pion_quantile_require_has_pion=False,
        pion_bin_edge_method="equal_frequency",
    )

    y_true_cc1pi = np.isin(pid, cc1pi_classes).astype(int)
    y_pred_cc1pi = (
        (n_muons == 1) & (n_charged_prongs == 1) & (improved_nmichel == 1)
    ).astype(int)

    tp = np.sum((y_pred_cc1pi == 1) & (y_true_cc1pi == 1))
    fp = np.sum((y_pred_cc1pi == 1) & (y_true_cc1pi == 0))
    fn = np.sum((y_pred_cc1pi == 0) & (y_true_cc1pi == 1))
    tn = np.sum((y_pred_cc1pi == 0) & (y_true_cc1pi == 0))

    baseline_fpr_cc1pi = fp / (fp + tn)
    baseline_recall_cc1pi = tp / (tp + fn)
    baseline_precision_cc1pi = tp / (tp + fp)
    print(f"[{playlist}] CC1π± baseline: "
          f"Precision={baseline_precision_cc1pi:.4f}  "
          f"Recall={baseline_recall_cc1pi:.4f}  "
          f"FPR={baseline_fpr_cc1pi:.4f}")


    # --- Recompute model metrics with the baseline's FPR ---
    metrics_cc1pi = compute_all_metrics(
        results, data_cc1pi, signal_classes=cc1pi_classes, fixed_fpr=[baseline_fpr_cc1pi],
        playlist=playlist,
        pion_bins_require_has_pion=False,
    )
    baseline_cc1pi = compute_signal_baseline(
        results, data_cc1pi, signal_classes=cc1pi_classes, playlist=playlist,
        pion_bins_require_has_pion=False,
    )

    # --- Per-bin baseline recall for E and theta ---
    is_signal_cc1pi = y_true_cc1pi == 1
    reco_baseline_tpr_cc1pi = {
        "E": compute_reco_baseline_recall_per_bin(
            y_pred_cc1pi, is_signal_cc1pi,
            data_cc1pi["pion_E_MC"], data_cc1pi["pion_E_MC_bins"],
            has_pion=None,
        ),
        "theta": compute_reco_baseline_recall_per_bin(
            y_pred_cc1pi, is_signal_cc1pi,
            data_cc1pi["pion_theta_MC"], data_cc1pi["pion_theta_MC_bins"],
            has_pion=None,
            finite_bin_var=True,
        ),
    }

    # --- Standard kinematic plots (with baseline on TPR panel) ---
    fig = plot_cc1pi_vs_pion_kinematics(
        metrics_cc1pi, data_cc1pi, baseline_cc1pi, uncertainties=True,
        fixed_fpr=[baseline_fpr_cc1pi],
        reco_baseline_tpr=reco_baseline_tpr_cc1pi,
        colors=clrs_dict_full,
        playlist=playlist,
    )
    figs_cc1pi.append(fig)
    fig.show()

    fig = plot_prc_curves(
        results, signal_classes=cc1pi_classes,
        title=fr"PRC — $CC1\pi^\pm$ tagging - Minerva Open Data Playlist {playlist}",
        playlist=playlist,
        colors=clrs_dict_full,
        uncertainties=True,
    )
    figs_cc1pi.append(fig)
    fig.show()

    fig = plot_binned_by_inttype(
        results, data_cc1pi,
        signal_classes=cc1pi_classes,
        x_var="pion_E",
        xlabel=r"True $E_\pi$ [GeV]",
        title=fr"$CC1\pi^\pm$ tagging - Minerva Open Data Playlist {playlist} - by interaction type",
        log_x=True,
        uncertainties=True,
        fixed_fpr=[baseline_fpr_cc1pi],
        reco_baseline_pred=y_pred_cc1pi,
        playlist=playlist,
        colors=clrs_dict_full,
        pion_bins_require_has_pion=False,
    )
    figs_cc1pi.append(fig)
    fig.show()

    fig = plot_binned_by_inttype(
        results, data_cc1pi,
        signal_classes=cc1pi_classes,
        x_var="pion_theta",
        xlabel=r"True $\theta_\pi$ [rad]",
        title=fr"$CC1\pi^\pm$ tagging - Minerva Open Data Playlist {playlist} - by interaction type",
        uncertainties=True,
        fixed_fpr=[baseline_fpr_cc1pi],
        reco_baseline_pred=y_pred_cc1pi,
        playlist=playlist,
        colors=clrs_dict_full,
        pion_bins_require_has_pion=False,
    )
    figs_cc1pi.append(fig)
    fig.show()

    save_figures_to_pdf(figs_cc1pi, OUTPUT_DIR / f"eval_cc1pi_tagging_{playlist}.pdf")


In [ ]:
fig = plot_prc_curves(
    results, signal_classes=cc1pi_classes,
    title=fr"PRC — $CC1\pi^\pm$ tagging - Minerva Open Data Playlist {PLAYLISTS[0]}",
    playlist=PLAYLISTS[0],
    uncertainties=True,
    colors=clrs_dict_full,
)
figs_cc1pi.append(fig)
fig.show()

In [ ]:
for playlist in PLAYLISTS:
    data = data_by_playlist[playlist]
    figs_cc1pi_q3 = []

    test_idx = data["test_idx"][playlist]
    baselines_pl = data["baselines"][playlist]
    n_muons = baselines_pl["n_muons"][test_idx]
    n_charged_prongs = baselines_pl["n_charged_prongs"][test_idx]
    improved_nmichel = baselines_pl["improved_nmichel"][test_idx]
    first_model = next(iter(results))
    run0 = results[first_model][0][playlist]
    pid = run0["pid"]

    y_true_cc1pi = np.isin(pid, cc1pi_classes).astype(int)
    y_pred_cc1pi = (
        (n_muons == 1) & (n_charged_prongs == 1) & (improved_nmichel == 1)
    ).astype(int)

    tp = np.sum((y_pred_cc1pi == 1) & (y_true_cc1pi == 1))
    fp = np.sum((y_pred_cc1pi == 1) & (y_true_cc1pi == 0))
    fn = np.sum((y_pred_cc1pi == 0) & (y_true_cc1pi == 1))
    tn = np.sum((y_pred_cc1pi == 0) & (y_true_cc1pi == 0))
    baseline_fpr_cc1pi = fp / (fp + tn)


    metrics_q3_cc1pi = compute_all_metrics_q3(
        results, data, signal_classes=cc1pi_classes, fixed_fpr=[baseline_fpr_cc1pi],
        playlist=playlist,
    )
    baseline_cc1pi = compute_signal_baseline(results, data, signal_classes=cc1pi_classes, playlist=playlist)

    is_signal_cc1pi = y_true_cc1pi == 1
    reco_baseline_tpr_q3_cc1pi = compute_reco_baseline_recall_per_bin(
        y_pred_cc1pi, is_signal_cc1pi, data["q3_GeV"], data["q3_bin_edges"],
    )

    fig = plot_multi_pion_vs_q3(
        metrics_q3_cc1pi, data, baseline_cc1pi["q3"],
        fixed_fpr=[baseline_fpr_cc1pi], uncertainties=True,
        reco_baseline_tpr_q3=reco_baseline_tpr_q3_cc1pi,
        colors=clrs_dict_full,
        title=fr"$CC1\pi^\pm$ tagging - Minerva Open Data Playlist {playlist}",
        playlist=playlist,
    )
    figs_cc1pi_q3.append(fig)
    fig.show()

    fig = plot_binned_by_inttype(
        results, data,
        signal_classes=cc1pi_classes,
        x_var="q3",
        xlabel=r"True $q_3$ [GeV]",
        title=fr"$CC1\pi^\pm$ tagging - Minerva Open Data Playlist {playlist} - by interaction type",
        uncertainties=True,
        fixed_fpr=[baseline_fpr_cc1pi],
        reco_baseline_pred=y_pred_cc1pi,
        playlist=playlist,
        colors=clrs_dict_full,
    )
    figs_cc1pi_q3.append(fig)
    fig.show()

    save_figures_to_pdf(figs_cc1pi_q3, OUTPUT_DIR / f"eval_cc1pi_tagging_q3_{playlist}.pdf")

## Multi-pion tagging vs q₃

In [ ]:
multi_pi_classes = [0, 1]

for playlist in PLAYLISTS:
    data = data_by_playlist[playlist]
    figs_npi = []

    test_idx = data["test_idx"][playlist]
    baselines_pl = data["baselines"][playlist]
    n_muons = baselines_pl["n_muons"][test_idx]
    n_charged_prongs = baselines_pl["n_charged_prongs"][test_idx]
    improved_nmichel = baselines_pl["improved_nmichel"][test_idx]
    first_model = next(iter(results))
    run0 = results[first_model][0][playlist]
    pid = run0["pid"]

    # --- Reconstruction baseline ---
    y_true_ccnpi = np.isin(pid, multi_pi_classes).astype(int)
    y_pred_ccnpi = (
        (n_muons == 1) & (n_charged_prongs >= 1) & (improved_nmichel >= 1)
    ).astype(int)

    tp = np.sum((y_pred_ccnpi == 1) & (y_true_ccnpi == 1))
    fp = np.sum((y_pred_ccnpi == 1) & (y_true_ccnpi == 0))
    fn = np.sum((y_pred_ccnpi == 0) & (y_true_ccnpi == 1))
    tn = np.sum((y_pred_ccnpi == 0) & (y_true_ccnpi == 0))
    baseline_fpr_ccnpi = fp / (fp + tn)
    baseline_recall_ccnpi = tp / (tp + fn)
    baseline_precision_ccnpi = tp / (tp + fp)
    print(f"[{playlist}] CCNπ baseline: "
          f"Precision={baseline_precision_ccnpi:.4f}  "
          f"Recall={baseline_recall_ccnpi:.4f}  "
          f"FPR={baseline_fpr_ccnpi:.4f}")


    # --- Recompute model metrics with the baseline's FPR ---
    metrics_q3 = compute_all_metrics_q3(
        results, data, signal_classes=multi_pi_classes, fixed_fpr=[baseline_fpr_ccnpi],
        playlist=playlist
    )
    baseline_multi = compute_signal_baseline(results, data, signal_classes=multi_pi_classes, playlist=playlist)

    # --- Per-bin baseline recall for q3 ---
    is_signal_ccnpi = y_true_ccnpi == 1
    reco_baseline_tpr_q3 = compute_reco_baseline_recall_per_bin(
        y_pred_ccnpi, is_signal_ccnpi, data["q3_GeV"], data["q3_bin_edges"],
    )

    fig = plot_multi_pion_vs_q3(
        metrics_q3, data, baseline_multi["q3"],
        fixed_fpr=[baseline_fpr_ccnpi], uncertainties=True,
        reco_baseline_tpr_q3=reco_baseline_tpr_q3,
        colors=clrs_dict_full,
        title=fr"$CCN\pi^\pm$ tagging ($N \geq 1$) - Minerva Open Data Playlist {playlist}",
        playlist=playlist,
    )
    figs_npi.append(fig)
    fig.show()

    fig = plot_prc_curves(
        results, signal_classes=multi_pi_classes,
        title=fr"PRC — $CCN\pi^\pm$ tagging ($N \geq 1$) - Minerva Open Data Playlist {playlist}",
        playlist=playlist,
        uncertainties=True,
        colors=clrs_dict_full,
    )
    figs_npi.append(fig)
    fig.show()

    fig = plot_binned_by_inttype(
        results, data,
        signal_classes=multi_pi_classes,
        x_var="q3",
        xlabel=r"True $q_3$ [GeV]",
        title=fr"$CCN\pi^\pm$ tagging ($N \geq 1$) - Minerva Open Data Playlist {playlist} - by interaction type",
        uncertainties=True,
        fixed_fpr=[baseline_fpr_ccnpi],
        reco_baseline_pred=y_pred_ccnpi,
        playlist=playlist,
        colors=clrs_dict_full,
    )
    figs_npi.append(fig)
    fig.show()

    save_figures_to_pdf(figs_npi, OUTPUT_DIR / f"eval_Npi_tagging_{playlist}.pdf")

## CCπ⁰ tagging


In [ ]:
cc1pi0_classes = [2]
PI0_MASS = 134.977  # MeV
DELTA_M = PI0_MASS

for playlist in PLAYLISTS:
    data = data_by_playlist[playlist]
    figs_pi0 = []

    test_idx = data["test_idx"][playlist]
    baselines_pl = data["baselines"][playlist]
    n_muons = baselines_pl["n_muons"][test_idx]
    is_pizero_signal = baselines_pl["is_pizero_signal"][test_idx]
    two_gamma_inv_mass = baselines_pl["two_gamma_invariant_mass"][test_idx]
    n_michel = baselines_pl["improved_nmichel"][test_idx]

    first_model = next(iter(results))
    run0 = results[first_model][0][playlist]
    pid = run0["pid"]
    data_pi0 = data_with_signal_pion_bins(
        data, pid, cc1pi0_classes,
        pion_quantile_require_has_pion=False,
        pion_bin_edge_method="equal_frequency",
    )
    y_true_pi0 = np.isin(pid, cc1pi0_classes).astype(int)

    y_pred_baseline = (
        (n_muons == 1) &
        (is_pizero_signal == 2) &
        (np.abs(two_gamma_inv_mass - PI0_MASS) < DELTA_M) &
        (n_michel == 0)
    ).astype(int)

    tp_g = np.sum((y_pred_baseline == 1) & (y_true_pi0 == 1))
    fp_g = np.sum((y_pred_baseline == 1) & (y_true_pi0 == 0))
    fn_g = np.sum((y_pred_baseline == 0) & (y_true_pi0 == 1))
    tn_g = np.sum((y_pred_baseline == 0) & (y_true_pi0 == 0))
    baseline_fpr = fp_g / (fp_g + tn_g)
    baseline_recall_global = tp_g / (tp_g + fn_g)
    baseline_precision_global = tp_g / (tp_g + fp_g)
    print(f"[{playlist}] CCπ0 baseline (Δm = {DELTA_M:.0f} MeV): "
          f"Precision={baseline_precision_global:.4f}  "
          f"Recall={baseline_recall_global:.4f}  "
          f"FPR={baseline_fpr:.4f}")

    metrics_cc1pi0 = compute_all_metrics(
        results, data_pi0, signal_classes=cc1pi0_classes, fixed_fpr=[baseline_fpr],
        playlist=playlist,
        pion_bins_require_has_pion=False,
    )
    baseline_cc1pi0 = compute_signal_baseline(
        results, data_pi0, signal_classes=cc1pi0_classes, playlist=playlist,
        pion_bins_require_has_pion=False,
    )

    is_signal_pi0 = y_true_pi0 == 1
    reco_baseline_tpr = {
        "E": compute_reco_baseline_recall_per_bin(
            y_pred_baseline, is_signal_pi0,
            data_pi0["pion_E_MC"], data_pi0["pion_E_MC_bins"],
            has_pion=None,
        ),
        "theta": compute_reco_baseline_recall_per_bin(
            y_pred_baseline, is_signal_pi0,
            data_pi0["pion_theta_MC"], data_pi0["pion_theta_MC_bins"],
            has_pion=None,
            finite_bin_var=True,
        ),
    }

    fig = plot_cc1pi_vs_pion_kinematics(
        metrics_cc1pi0, data_pi0, baseline_cc1pi0, uncertainties=True,
        fixed_fpr=[baseline_fpr],
        reco_baseline_tpr=reco_baseline_tpr,
        colors=clrs_dict_full,
        suptitle=fr"$CC1\pi^0$ tagging - Minerva Open Data Playlist {playlist}",
        playlist=playlist,
    )
    figs_pi0.append(fig)
    fig.show()

    fig2 = plot_binned_by_inttype(
        results, data_pi0, signal_classes=cc1pi0_classes,
        x_var="pion_E", xlabel=r"True $E_\pi$ [GeV]",
        title=fr"$CC1\pi^0$ tagging - Minerva Open Data Playlist {playlist} - by interaction type",
        log_x=True, uncertainties=True,
        fixed_fpr=[baseline_fpr],
        reco_baseline_pred=y_pred_baseline,
        playlist=playlist,
        colors=clrs_dict_full,
        pion_bins_require_has_pion=False,
    )
    figs_pi0.append(fig2)
    fig2.show()

    fig3 = plot_binned_by_inttype(
        results, data_pi0, signal_classes=cc1pi0_classes,
        x_var="pion_theta", xlabel=r"True $\theta_\pi$ [rad]",
        title=fr"$CC1\pi^0$ tagging - Minerva Open Data Playlist {playlist} - by interaction type",
        uncertainties=True,
        fixed_fpr=[baseline_fpr],
        reco_baseline_pred=y_pred_baseline,
        playlist=playlist,
        colors=clrs_dict_full,
        pion_bins_require_has_pion=False,
    )
    figs_pi0.append(fig3)
    fig3.show()

    fig4 = plot_prc_curves(
        results, signal_classes=cc1pi0_classes,
        title=fr"PRC — $CC1\pi^0$ tagging - Minerva Open Data Playlist {playlist}",
        playlist=playlist,
        uncertainties=True,
        colors=clrs_dict_full,
    )
    figs_pi0.append(fig4)
    fig4.show()

    save_figures_to_pdf(figs_pi0, OUTPUT_DIR / f"eval_cc1pi0_tagging_{playlist}.pdf")

In [ ]:
PI0_MASS = 134.977  # MeV


def get_pi0_baseline_pred(is_pizero_signal, two_gamma_inv_mass, delta_m, n_muons, n_michel):
    """Reconstruction-level pi0 baseline for a given mass window."""
    has_candidate = is_pizero_signal == 2
    in_mass_window = np.abs(two_gamma_inv_mass - PI0_MASS) < delta_m
    return (
        (n_muons == 1) & has_candidate & in_mass_window & (n_michel == 0)
    ).astype(int)


def precision_recall_fpr_vs_deltam(y_true, is_pizero_signal, two_gamma_inv_mass,
                                   delta_m_values, n_muons, n_michel):
    """Compute precision, recall, and FPR for each delta_m window."""
    precisions, recalls, fprs = [], [], []
    for dm in delta_m_values:
        y_pred = get_pi0_baseline_pred(is_pizero_signal, two_gamma_inv_mass, dm, n_muons, n_michel)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        tn = np.sum((y_pred == 0) & (y_true == 0))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        precisions.append(precision)
        recalls.append(recall)
        fprs.append(fpr)
    return np.array(precisions), np.array(recalls), np.array(fprs)


delta_m_values = np.linspace(1, 509, 20)

for playlist in PLAYLISTS:
    data = data_by_playlist[playlist]
    test_idx = data["test_idx"][playlist]
    baselines_pl = data["baselines"][playlist]
    is_pizero_signal = baselines_pl["is_pizero_signal"][test_idx]
    two_gamma_inv_mass = baselines_pl["two_gamma_invariant_mass"][test_idx]
    n_muons = baselines_pl["n_muons"][test_idx]
    n_michel = baselines_pl["improved_nmichel"][test_idx]
    first_model = next(iter(results))
    run0 = results[first_model][0][playlist]
    pid = run0["pid"]
    y_true = np.isin(pid, cc1pi0_classes).astype(int)

    precisions, recalls, fprs = precision_recall_fpr_vs_deltam(
        y_true, is_pizero_signal, two_gamma_inv_mass, delta_m_values, n_muons, n_michel
    )
    n_true_signal = y_true.sum()
    n_total = len(y_true)
    sbr = n_true_signal / (n_total - n_true_signal)
    print(f"[{playlist}] True CC1π0 signal events: {n_true_signal:,} / {n_total:,}  "
          f"(signal fraction = {n_true_signal / n_total:.3f},  S/B = {sbr:.4f})")

    fig, ax1 = plt.subplots(figsize=(10, 5))
    color_p = "steelblue"
    color_r = "darkorange"
    color_f = "firebrick"
    ax1.plot(delta_m_values, precisions, ".-", color=color_p, linewidth=1.5, label="Precision")
    ax1.plot(delta_m_values, recalls, ".-", color=color_r, linewidth=1.5, label="Recall")
    ax1.plot(delta_m_values, fprs, ".-", color=color_f, linewidth=1.5, label="FPR")
    ax1.set_xlabel(r"$\Delta m$ window [MeV]")
    ax1.set_ylabel(r"Fraction")
    ax1.legend(loc="center right")
    ax1.grid(True, alpha=0.3)
    ax1.set_title(r"$CC\pi^0$ baseline vs. $\Delta m$ window")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"pi0_baseline_deltam_{playlist}.pdf", bbox_inches="tight")
    fig.show()